## BioPAX abstraction: Pathway-centered view

In [10]:
from SPARQLWrapper import SPARQLWrapper, JSON, CSV, N3, XML, TURTLE
import subprocess
import time
import os
import IPython
import pandas as pd

In [11]:
endpoint_reactome = "http://localhost:3030/reactome"
rdfFormat = "turtle"
current_directory = os.getcwd()
BioPAX_Ontology_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'Homo_sapiens_v94.owl')

In [12]:
def displaySparqlResults(results):
    '''
    Displays as HTML the result of a SPARQLWrapper query in a Jupyter notebook.
    
        Parameters:
            results (dictionnary): the result of a call to SPARQLWrapper.query().convert()
    '''
    variableNames = results['head']['vars']
    tableCode = '<table><tr><th>{}</th></tr><tr>{}</tr></table>'.format('</th><th>'.join(variableNames), '</tr><tr>'.join('<td>{}</td>'.format('</td><td>'.join([row[vName]['value'] if vName in row.keys() else "&nbsp;" for vName in variableNames]))for row in results["results"]["bindings"]))
    IPython.display.display(IPython.display.HTML(tableCode))

In [13]:
prefixes = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/>
PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
PREFIX reactome: <http://www.reactome.org/biopax/40/48887#>
PREFIX abstraction:<http://abstraction/#>
"""

In [14]:
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', ReactomeBioPAX_file_path,
    '--file', BioPAX_Ontology_file_path,
    '/reactome']

process = subprocess.Popen(command)
time.sleep(60)

13:52:14 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/../BioPAXData/Homo_sapiens_v94.owl
13:52:15 WARN  riot            :: [line: 66845, col: 48] {W137} Input is large. Switching off checking for illegal reuse of rdf:ID's.
13:52:35 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/../BioPAXData/biopax-level3.owl
13:52:35 INFO  Server          :: Running in read-only mode for /reactome
13:52:35 INFO  Server          :: Apache Jena Fuseki 4.9.0
13:52:35 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
13:52:35 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/run
13:52:35 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/run/shiro.ini
13:52:36 INFO  Server          :: Database: in-memory, with files loaded
1

### SPARQL queries for abstraction

#### 1 - IsAChildOf

Defines a relation that decribes the direct hierarchy of pathways.

In [ ]:
start_time = time.time()

query_is_a_child_of = """
SELECT DISTINCT ?subPathwayID ?pathwayID
WHERE {
  ?pathway rdf:type bp3:Pathway .
  ?pathway bp3:pathwayComponent ?subPathway .
  ?subPathway rdf:type bp3:Pathway .

  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .
  
  ?subPathway bp3:xref ?subPathwayXref .
  ?subPathwayXref rdf:type bp3:UnificationXref .
  ?subPathwayXref bp3:db "Reactome" .
  ?subPathwayXref bp3:id ?subPathwayID .
}
"""

sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_is_a_child_of)

sparql.setReturnFormat(JSON)
results = sparql.query().convert()
#displaySparqlResults(results)

sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_IsAChildOf.csv", "wb") as f:
    f.write(results)

print("--- %s seconds ---" % (time.time() - start_time))

# add interaction type in dataframe
isachildof = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_IsAChildOf.csv", sep=",", header=0)
new_col = ["abstraction:IsAChildOf"]*len(isachildof)
isachildof.insert(loc=1, column="interaction", value=new_col)
print(isachildof.head())
isachildof.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_IsAChildOf.csv", sep=",", header=True, index=False)

15:15:49 INFO  Fuseki          :: [1] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A%0ASELECT+DISTINCT+%3FsubPathwayID+%3FpathwayID%0AWHERE+%7B%0A++%3Fpathway+rdf%3

--- 1.3639857769012451 seconds ---
    subPathwayID             interaction      pathwayID
0  R-HSA-8877627  abstraction:IsAChildOf  R-HSA-6806667
1  R-HSA-6806664  abstraction:IsAChildOf  R-HSA-6806667
2   R-HSA-975634  abstraction:IsAChildOf  R-HSA-6806667
3  R-HSA-1296041  abstraction:IsAChildOf  R-HSA-1296059
4   R-HSA-190241  abstraction:IsAChildOf  R-HSA-5654738


15:15:50 INFO  Fuseki          :: [2] 200 OK (569 ms)


#### 2 - NextStepPathway

Defines a relation that describes the sequence of pathway steps across different pathways

In [ ]:
start_time = time.time()

query_next_step_pathway = """ 
SELECT DISTINCT ?pathwayID ?nextPathwayID
WHERE {
  ?pathway rdf:type bp3:Pathway .
  ?nextPathway rdf:type bp3:Pathway .
  
  ?pathway bp3:pathwayOrder ?pathwayStep .
  ?nextPathway bp3:pathwayOrder ?nextStep .
  
  ?pathwayStep bp3:nextStep ?nextStep .
  
  FILTER (?pathway != ?nextPathway)

  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .
  
  ?nextPathway bp3:xref ?nextPathwayXref .
  ?nextPathwayXref rdf:type bp3:UnificationXref .
  ?nextPathwayXref bp3:db "Reactome" .
  ?nextPathwayXref bp3:id ?nextPathwayID .
}
"""

sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_next_step_pathway)

sparql.setReturnFormat(JSON)
results = sparql.query().convert()
#displaySparqlResults(results)

sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathway.csv", "wb") as f:
    f.write(results)

print("--- %s seconds ---" % (time.time() - start_time))

# add interaction type in dataframe
nextsteppathway = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathway.csv", sep=",", header=0)
new_col = ["abstraction:NextStepPathway"]*len(nextsteppathway)
nextsteppathway.insert(loc=1, column="interaction", value=new_col)
print(nextsteppathway.head())
nextsteppathway.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathway.csv", sep=",", header=True, index=False)

15:16:02 INFO  Fuseki          :: [3] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpathwayID+%3FnextPathwayID%0AWHERE+%7B%0A++%3Fpathway+rdf

--- 363.7645118236542 seconds ---
       pathwayID                  interaction  nextPathwayID
0  R-HSA-1474151  abstraction:NextStepPathway  R-HSA-9009391
1  R-HSA-1474151  abstraction:NextStepPathway   R-HSA-203615
2  R-HSA-1474151  abstraction:NextStepPathway  R-HSA-5218920
3   R-HSA-434313  abstraction:NextStepPathway   R-HSA-200425
4   R-HSA-174411  abstraction:NextStepPathway   R-HSA-174414


15:22:06 INFO  Fuseki          :: [4] 200 OK (181.255 s)


In [ ]:
# Remonter les next step pathways aux pathways parents
start_time = time.time()

query_next_step_pathway_to_parents = """ 
SELECT DISTINCT ?previousPathwayAncestorID ?nextPathwayAncestorID
WHERE {
  ?previousStep rdf:type bp3:PathwayStep .
  ?previousStep bp3:nextStep ?nextStep .
  
  ?previousPathway bp3:pathwayOrder ?previousStep .
  ?nextPathway bp3:pathwayOrder ?nextStep .
  
  ?previousPathwayAncestor bp3:pathwayComponent* ?previousPathway .
  ?nextPathwayAncestor bp3:pathwayComponent* ?nextPathway .
  
  FILTER NOT EXISTS {
    ?previousPathwayAncestor bp3:pathwayComponent* ?nextPathwayAncestor .
  }
  FILTER NOT EXISTS {
    ?nextPathwayAncestor bp3:pathwayComponent* ?previousPathwayAncestor .
  }
  
  ?previousPathwayAncestor bp3:xref [
      rdf:type bp3:UnificationXref ;
      bp3:db "Reactome" ;
      bp3:id ?previousPathwayAncestorID
  ] .
  
  ?nextPathwayAncestor bp3:xref [
      rdf:type bp3:UnificationXref ;
      bp3:db "Reactome" ;
      bp3:id ?nextPathwayAncestorID
  ] .
}
"""

sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_next_step_pathway_to_parents)
sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathwayToParents.csv", 'wb') as f:
    f.write(results)

print("--- %s seconds ---" % (time.time() - start_time))

# add interaction type in dataframe
nextsteppathwayancestors = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathwayToParents.csv", sep=",", header=0)
new_col = ["abstraction:NextStepPathway"]*len(nextsteppathwayancestors)
nextsteppathwayancestors.insert(loc=1, column="interaction", value=new_col)
print(nextsteppathwayancestors.head())
nextsteppathwayancestors.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathwayToParents.csv", sep=",", header=True, index=False)

11:43:23 INFO  Fuseki          :: [14] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpreviousPathwayAncestorID+%3FnextPathwayAncestorID%0AWHE

--- 437.71879720687866 seconds ---
  previousPathwayAncestorID                  interaction nextPathwayAncestorID
0             R-HSA-2467813  abstraction:NextStepPathway          R-HSA-174154
1             R-HSA-2467813  abstraction:NextStepPathway          R-HSA-176409
2             R-HSA-2467813  abstraction:NextStepPathway          R-HSA-176814
3             R-HSA-2467813  abstraction:NextStepPathway          R-HSA-174143
4             R-HSA-2467813  abstraction:NextStepPathway          R-HSA-453276


11:50:41 INFO  Fuseki          :: [14] 200 OK (437.710 s)


### Concatenation of output files

In [ ]:
# concatenate output files
q1 = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_IsAChildOf.csv", header=0, sep=",")
#new_col = [pd.NA]*len(q1)
#q1.insert(loc=3, column="tag", value=new_col)
q1 = q1.rename(columns={"subPathwayID": "pathway1", "interaction": "interaction", "pathwayID": "pathway2"})
#q1 = q1.drop(q1.index[0]).reset_index(drop=True)

q2 = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathway.csv", header=0, sep=",")
#new_col = [pd.NA]*len(q2)
#q2.insert(loc=3, column="tag", value=new_col)
q2 = q2.rename(columns={"pathwayID": "pathway1", "interaction": "interaction", "nextPathwayID": "pathway2"})
#q2 = q2.drop(q2.index[0]).reset_index(drop=True)

q3 = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_NextStepPathwayToParents.csv")
q3 = q3.rename(columns={"previousPathwayAncestorID": "pathway1", "interaction": "interaction", "nextPathwayAncestorID": "pathway2"})
#print(q3.head())
#q3 = q3.drop(q3.index[0]).reset_index(drop=True)

concat_df = pd.concat([q1, q2], ignore_index=True)
concat_df.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_PathwayAbstraction.csv", sep=",", header=True, index=False)
print(concat_df)

concat_df_to_ancestors = pd.concat([q1,q3], ignore_index=True)
concat_df_to_ancestors.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_PathwayAbstraction_NextStepsAncestors.csv", sep=",", header=True, index=False)

           pathway1                  interaction       pathway2
0     R-HSA-8877627       abstraction:IsAChildOf  R-HSA-6806667
1     R-HSA-6806664       abstraction:IsAChildOf  R-HSA-6806667
2      R-HSA-975634       abstraction:IsAChildOf  R-HSA-6806667
3     R-HSA-1296041       abstraction:IsAChildOf  R-HSA-1296059
4      R-HSA-190241       abstraction:IsAChildOf  R-HSA-5654738
...             ...                          ...            ...
6699   R-HSA-499943  abstraction:NextStepPathway    R-HSA-83936
6700   R-HSA-499943  abstraction:NextStepPathway  R-HSA-9818027
6701   R-HSA-167162  abstraction:NextStepPathway   R-HSA-167160
6702  R-HSA-2485179  abstraction:NextStepPathway  R-HSA-2514859
6703  R-HSA-9018896  abstraction:NextStepPathway   R-HSA-373076

[6704 rows x 3 columns]


In [18]:
# end process
process.kill()
time.sleep(60)